<a href="https://colab.research.google.com/github/jdasam/ant5015/blob/2025/notebooks/3rd_week_pitch_classification" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Synthesizing and Classifying Sine Waves

We start from scratch: generate audio with math, visualize it, and train a simple classifier.

## Sine Wave Synthesis

A sine wave at frequency $f$ (Hz) is:
$$x(t) = A \cdot \sin(2\pi f t)$$

Import the libraries, then build the wave sample-by-sample:
convert sample indices → seconds → radians → sine values.

In [ ]:
import torch
import matplotlib.pyplot as plt
import IPython.display as ipd
from math import pi

torch.set_printoptions(sci_mode=False)

sr = 16000        # sample rate (samples/sec)
frequency = 440   # Hz
amplitude = 0.5

# TODO: create sample_t with torch.arange
# TODO: convert to sec_t, compute accum_rad, then sin_value


Plot the first 500 samples to confirm the wave shape.

In [ ]:
plt.plot(sin_value[:500])


Listen to the result.

In [ ]:
ipd.Audio(sin_value, rate=sr)


Wrap the synthesis steps into a reusable function.

```python
def make_sine_wave(duration, sr, frequency, amplitude) -> torch.Tensor
```

In [ ]:
def make_sine_wave(duration, sr, frequency, amplitude):
    # TODO: implement
    pass

sine_wave = make_sine_wave(3, 16000, 340, 2)
ipd.Audio(sine_wave, rate=sr, normalize=True)


Save the waveform as a WAV file, then reload and plot it.

In [ ]:
import torchaudio

# TODO: torchaudio.save('test.wav', ...) — remember to add a channel dim with .unsqueeze(0)


In [ ]:
# TODO: load 'test.wav' and plot the first 1000 samples


## Time-Varying Frequency

When frequency changes over time, we can no longer multiply by a fixed $f$.
Instead, accumulate the instantaneous phase with `torch.cumsum`:

$$\phi[n] = \sum_{k=0}^{n} \frac{f[k] \cdot 2\pi}{sr}$$

First, build intuition for `torch.cumsum` with a small example.

In [ ]:
dummy_tensor = torch.arange(10)
print(dummy_tensor)
# TODO: compute torch.cumsum(dummy_tensor, dim=0)


Now implement `make_sine_wave_with_freq_seq(sr, freq_sequence, amplitude)` using `cumsum`, then create a vibrato effect by modulating frequency around 440 Hz.

In [ ]:
def make_sine_wave_with_freq_seq(sr, freq_sequence, amplitude):
    # TODO: accumulate phase with torch.cumsum, return torch.sin(...) * amplitude
    pass

freq_sequence = 440 + make_sine_wave(4, 16000, 4, 50)
plt.plot(freq_sequence)
# sine_with_vibrato = make_sine_wave_with_freq_seq(16000, freq_sequence, 0.5)
# ipd.Audio(sine_with_vibrato * 0.2, rate=16000, normalize=False)


## Synthesizer Class

Refactor the function into a `SineGenerator` class with:
- `__call__(frequency, amplitude)` — single sine wave
- `make_random_wav(frequency)` — sum of 1–10 harmonics with amplitude $\propto 1/\sqrt{i+1}$

In [ ]:
import random

class SineGenerator:
    def __init__(self, sr, duration):
        # TODO
        pass

    def __call__(self, frequency, amplitude):
        # TODO
        pass

    def make_random_wav(self, frequency):
        # TODO: random number of harmonics, decreasing amplitude
        pass

oscilator = SineGenerator(sr=16000, duration=4)
# ipd.Audio(oscilator(440, 0.3), rate=oscilator.sr)
# ipd.Audio(oscilator.make_random_wav(442) * 0.03, rate=oscilator.sr, normalize=False)


Sum the first three harmonics of f0 = 220 Hz and listen to the result.

In [ ]:
f0 = 220
f1 = f0 * 2
f2 = f0 * 3

# TODO: wav_harmonics = oscilator(f0, 0.5) + ...
# ipd.Audio(wav_harmonics * 0.2, rate=sr, normalize=False)


Plot `wav_sum` (the sum of two waves) to see constructive/destructive interference.

In [ ]:
plt.plot(wav_sum[:1000])


## Spectrogram Analysis

Convert a waveform to a 2-D time-frequency representation using the Short-Time Fourier Transform (STFT).
Use `torchaudio.transforms.Spectrogram` (n_fft=1024, hop_length=512) and `AmplitudeToDB`, then visualize.

In [ ]:
# TODO: create spec_converter and db_converter
# TODO: compute spec = db_converter(spec_converter(wav_harmonics))
# TODO: plt.imshow(spec, origin='lower', aspect='auto')


The spectrogram has shape `(F, T)`. Loop over time frames 45–54 and plot the first 200 frequency bins of each to see the spectral peaks.

In [ ]:
print(spec.shape)  # F x T
# TODO: loop over t_i in range(45, 55) and plot spec[:200, t_i]


## Building a Classifier

Generate 1000 random waveforms for each of four pitch classes `[220, 440, 660, 215]` using `oscilator.make_random_wav`. Store each `(wav, frequency_label)` pair in `training_data`.

In [ ]:
from tqdm.auto import tqdm

freq_classes = [220, 440, 660, 215]
num_samples = 1000
training_data = []

# TODO: for each freq, generate num_samples random wavs and append (wav, freq) to training_data


Implement a `Dataset` class that converts each `(wav, label)` pair into a `(spectrum_vector, class_idx)` pair on the fly:
- compute dB spectrogram, extract the middle time frame, divide by 80
- convert frequency label to class index using `self.class_name.index(...)`

In [ ]:
class Dataset:
    def __init__(self, data):
        # TODO: store data, collect sorted unique labels, create spec/db converters
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        # TODO: spectrogram -> middle frame / 80 -> (spectrum, class_idx)
        pass

dataset = Dataset(training_data)
dataset[0]


Wrap in a `DataLoader` and pull out one batch.

In [ ]:
data_loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

# TODO: iterate once and unpack the first batch into (spectrum, label)


In [ ]:
spectrum.shape


Define a `nn.Linear` model (513 → 4) and compute cross-entropy loss **manually**:
logit → softmax → probability of correct class → negative log-likelihood → mean.

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=513, out_features=4)
optimizer = torch.optim.Adam(model.parameters())

# TODO: logit = model(spectrum)
# TODO: prob -> prob_of_correct_class -> NLL -> loss


Run the full training loop for 2 epochs. For each batch: forward → loss → `loss.backward()` → `optimizer.step()` → `optimizer.zero_grad()`.

In [ ]:
loss_record = []
num_epochs = 2

for epoch in range(num_epochs):
    for batch in tqdm(data_loader):
        spectrum, label = batch
        # TODO: compute loss, backprop, update, record
        pass


Plot the loss curve and inspect the learned weight matrix.

In [ ]:
plt.plot(loss_record)


In [ ]:
model.weight.shape


In [ ]:
for idx in range(2):
    plt.plot(model.weight[idx].data[:200])


---
# Part 2: Pitch Classification on the NSynth Dataset

We now solve the same classification problem on **real instrument recordings** from the [NSynth dataset](https://magenta.tensorflow.org/datasets/nsynth). The key differences:
- Input: real audio instead of synthetic sine waves
- Feature: **Mel spectrogram** (log-frequency scale) instead of linear spectrogram
- Output: ~120 MIDI pitch classes instead of 4

## Dataset Setup

Download the NSynth test split (~300 MB) and extract it. The archive contains a `nsynth-test/audio/` folder and an `examples.json` metadata file.

In [ ]:
# TODO: !wget http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-test.jsonwav.tar.gz
# TODO: !tar -xf nsynth-test.jsonwav.tar.gz


Load `examples.json` into `meta`. Each key is a filename stem; each value is a dict with fields like `pitch`, `instrument_family`, `velocity`. Print the total count and inspect one entry.

In [ ]:
import json
from pathlib import Path

# TODO: meta = json.load(open('nsynth-test/examples.json'))
# TODO: print(len(meta)) and inspect list(meta.keys())[:5] and one full entry


Build `audio_fns`, a list of `Path` objects pointing to each WAV file. Assert every file exists, then load and listen to one example.

In [ ]:
import torchaudio
import IPython.display as ipd

audio_dir = Path('nsynth-test/audio')
fns = sorted(meta.keys())

# TODO: build audio_fns = [(audio_dir / fn).with_suffix('.wav') for fn in fns]
# TODO: assert every path exists
# TODO: load one file and display


## Mel Spectrogram

A regular spectrogram has **linearly** spaced frequency bins. The Mel scale compresses high frequencies (where pitch differences are less audible), mapping 513 linear bins down to ~80 Mel bins.

Inspect the filterbank matrix of `torchaudio.transforms.MelScale` (shape: `n_stft × n_mels`): visualize it with `plt.imshow` and overlay a few individual filters.

In [ ]:
import matplotlib.pyplot as plt

# TODO: mel_scale = torchaudio.transforms.MelScale(n_mels=80, sample_rate=16000,
#                                                   f_min=20, f_max=4000, n_stft=513)
# TODO: print shape, imshow filterbank, plot a few columns


Use `torchaudio.transforms.MelSpectrogram` + `AmplitudeToDB` to compute the Mel-dB spectrogram for one audio file and visualize it.

In [ ]:
# TODO: mel_db_conv = ... (chain MelSpectrogram and AmplitudeToDB)
# TODO: load one file, compute mel-dB spec, plt.imshow(..., origin='lower', aspect='auto')


## Classification

Collect all unique pitch values from `meta`, sort them into `total_pitches`, and print the range and count.

In [ ]:
# TODO: total_pitches = sorted(set(v['pitch'] for v in meta.values()))
# TODO: print min, max, len


Shuffle `audio_fns` with `random.seed(0)` for reproducibility, then split into `train_fns` (first 3000) and `test_fns` (rest).

In [ ]:
import random

# TODO: sort, shuffle, and split audio_fns


Implement `NsynthDataset`. Its `__getitem__` should:
1. Load the WAV file
2. Compute the Mel-dB spectrogram
3. Extract the middle time frame and divide by 80
4. Look up the pitch in `meta` using the file stem, convert to a class index
5. Return `(spectrum_vector, class_idx)`

In [ ]:
class NsynthDataset:
    def __init__(self, file_list, meta, pitch_classes):
        # TODO
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        # TODO
        pass

train_dataset = NsynthDataset(train_fns, meta, total_pitches)
test_dataset  = NsynthDataset(test_fns,  meta, total_pitches)
train_dataset[0]


Create `train_loader` (batch_size=50, shuffle=True) and `test_loader` (shuffle=False), then define a `nn.Linear(80 → len(total_pitches))` model and an Adam optimizer.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# TODO: train_loader and test_loader

torch.manual_seed(0)
# TODO: model = nn.Linear(...) and optimizer


Train for 10 epochs. Use `F.cross_entropy(logit, label)` for the loss. Print the average loss after each epoch.

In [ ]:
loss_record = []
num_epochs = 10

for epoch in range(num_epochs):
    epoch_losses = []
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        spectrum, label = batch
        # TODO: loss, backward, step, zero_grad
        pass
    # TODO: print average epoch loss


Plot the training loss and evaluate accuracy on the test set.

In [ ]:
plt.plot(loss_record)


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(test_loader):
        spectrum, label = batch
        # TODO: predict with argmax, accumulate correct and total
        pass

print(f'Test accuracy: {correct / total * 100:.2f}%')


Visualize the learned weight matrix — each row is an 80-dim Mel spectral template for one pitch class. Then plot individual weight vectors for a few pitches and observe how the peaks shift with pitch.

In [ ]:
plt.figure(figsize=(12, 6))
# TODO: plt.imshow(model.weight.detach())


In [ ]:
for pitch_idx in [0, 20, 40, 60]:
    # TODO: plt.plot(model.weight[pitch_idx].detach(), label=total_pitches[pitch_idx])
    pass
plt.legend()
plt.xlabel('Mel bin')
